# Pocket-Agent — Training Notebook
**Run this notebook on Colab T4 GPU (Runtime → Change runtime type → T4 GPU)**

This notebook:
1. Installs dependencies
2. Clones the repo
3. Generates synthetic training data
4. Fine-tunes `Qwen2.5-0.5B-Instruct` with QLoRA
5. Quantizes to GGUF Q3_K_M
6. Evaluates on built-in test suite
7. Uploads artifacts to HuggingFace Hub

In [30]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
print(result.stdout[:500])

Sat Apr 18 07:07:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


In [31]:
# ── Cell 2: Install training dependencies ────────────────────────────────────
# This takes ~2 minutes
%pip install -q \
    'torch>=2.2.0' \
    'transformers>=4.44.0' \
    'peft>=0.12.0' \
    'trl>=0.12.0' \
    'datasets>=2.19.0' \
    'bitsandbytes>=0.43.0' \
    'accelerate>=0.29.0' \
    'huggingface_hub>=0.20.0'

print('Dependencies installed ✅')

Dependencies installed ✅


In [32]:
# ── Cell 3: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/HaseebUllahButt/Vyro2'

if not os.path.exists('Vyro2'):
    !git clone {REPO_URL}

%cd Vyro2
print('Working directory:', os.getcwd())


Cloning into 'Vyro2'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 57 (delta 22), reused 45 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 44.85 KiB | 6.41 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/Vyro2/Vyro2/Vyro2
Working directory: /content/Vyro2/Vyro2/Vyro2


In [33]:
# ── Cell 4: Generate training data ───────────────────────────────────────────
!python src/data/generate.py
!python src/data/lint.py

# Verify
!wc -l data/train_clean.jsonl

No public_test.jsonl found — skipping hash protection (safe for synthetic data)
Data generation complete:
  weather                            : 50
  currency                           : 50
  convert                            : 40
  calendar                           : 40
  sql                                : 20
  refusals                           : 58
  multiturn                          : 14
  multilingual                       : 114
  adversarial (2x oversampled)       : 107
  refusals (oversampled)             : 103
  TOTAL after oversampling           : 703

Saved 703 examples → data/train.jsonl
Lint results: 703 passed, 0 failed

Clean dataset: 703 examples → data/train_clean.jsonl
✅ Lint passed — data quality looks good.
703 data/train_clean.jsonl


In [48]:
# Pull the fix from GitHub
!git pull origin main
print("Updated ✅")


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 504 bytes | 504.00 KiB/s, done.
From https://github.com/HaseebUllahButt/Vyro2
 * branch            main       -> FETCH_HEAD
   8560daa..009f65d  main       -> origin/main
Updating 8560daa..009f65d
Fast-forward
 notebooks/01_train.ipynb | 6 +++---
 1 file changed, 3 insertions(+), 3 deletions(-)
Updated ✅


In [35]:
# ── Cell 5: Pull latest fixes + Fine-tune ────────────────────────────────────
!git pull origin main
!python src/train/sft_lora.py

import os
if os.path.exists('artifacts/lora_adapter'):
    print('Adapter files:', os.listdir('artifacts/lora_adapter'))
else:
    print('ERROR: Training failed — check output above')


From https://github.com/HaseebUllahButt/Vyro2
 * branch            main       -> FETCH_HEAD
Already up to date.
Loading base model: Qwen/Qwen2.5-0.5B-Instruct
Loading weights: 100% 290/290 [00:00<00:00, 423.33it/s, Materializing param=model.norm.weight]                              mlp.up_proj.weight]
All bfloat16 params cast to float16 ✅
Generating train split: 703 examples [00:00, 127722.24 examples/s]
Map: 100% 703/703 [00:00<00:00, 7749.92 examples/s]
Training on 703 examples
Sample:
 <|im_start|>system
Today is 2026-04-18. You are an offline mobile assistant.
Emit tool calls ONLY as: <tool_call>{"tool": "name", "args": {...}}</tool_call>
Available tools:
  weather:  {location: string, unit: C|F}
  calendar: {action: list|create, date: YYYY-MM-DD, title?: string}
  convert:  {val
TRL version: 1.2.0
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to trai

In [ ]:
# ── Cell 6: Build llama.cpp and quantize ─────────────────────────────────────
import os

# Clone llama.cpp
if not os.path.exists('llama.cpp'):
    !git clone https://github.com/ggml-org/llama.cpp --depth=1

# Install Python deps for conversion
%pip install -q gguf sentencepiece

# Build quantize binary (~3 min)
!cd llama.cpp && cmake -B build && cmake --build build --config Release --target llama-quantize -j2

# Run full pipeline
!bash src/quantize/quantize.sh


 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
════════════════════════════════════════
 Step 1: Merge LoRA adapter into base weights
════════════════════════════════════════
Loading base model (CPU fp16)…
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 361.97it/s, Materializing param=model.norm.weight]                              
Loading LoRA adapter…
Merging and unloading…
Writing model shards: 100% 1/1 [00:04<00:00,  4.04s/it]
Merge complete ✅

════════════════════════════════════════
 Step 2: Convert to GGUF (fp16)
════════════════════════════════════════
INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16,

In [47]:
# ── Cell 7: Evaluate on built-in test suite ───────────────────────────────────
!python src/eval/score.py --verbose

# If score < 0.75, retrain one more epoch:
# !python src/train/sft_lora.py  (checkpoint resumes automatically)

Traceback (most recent call last):
  File "/content/Vyro2/Vyro2/Vyro2/src/eval/score.py", line 21, in <module>
    from inference import run
  File "/content/Vyro2/Vyro2/Vyro2/inference.py", line 13, in <module>
    from llama_cpp import Llama
ModuleNotFoundError: No module named 'llama_cpp'


In [38]:
HF_TOKEN = ''          # ← PASTE YOUR HF WRITE TOKEN
HF_REPO  = 'Haseeb9009/Vyrothon'          # ← e.g. 'yourusername/pocket-agent'

if not HF_TOKEN or not HF_REPO:
    print('⚠️  Set HF_TOKEN and HF_REPO above before running this cell')
else:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)
    api = HfApi()

    # Create repo
    api.create_repo(HF_REPO, repo_type='model', exist_ok=True)

    # Upload GGUF model
    print('Uploading model.gguf (~220MB)...')
    api.upload_file(
        path_or_fileobj='artifacts/model.gguf',
        path_in_repo='model.gguf',
        repo_id=HF_REPO,
        repo_type='model',
    )

    # Upload LoRA adapter
    print('Uploading lora_adapter/...')
    api.upload_folder(
        folder_path='artifacts/lora_adapter',
        path_in_repo='lora_adapter',
        repo_id=HF_REPO,
        repo_type='model',
    )

    print(f'\nDone! Model live at: https://huggingface.co/{HF_REPO}')
    print('Copy this URL into README.md and 02_judge_demo.ipynb')

Uploading model.gguf (~220MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


ValueError: Provided path: 'artifacts/model.gguf' is not a file on the local file system

In [ ]:
# ── Cell 9: Final gate checks before pushing ──────────────────────────────────
import os, time, sys
sys.path.insert(0, '.')

# Gate 1: No network imports
import subprocess
r = subprocess.run(
    ['grep', '-n', 'import requests\\|import urllib\\|import http\\|import socket', 'inference.py'],
    capture_output=True, text=True
)
print('Gate 1 - Network imports:', 'PASS ✅' if not r.stdout.strip() else f'FAIL ❌ {r.stdout}')

# Gate 2: Model size
size = os.path.getsize('artifacts/model.gguf') / 1e6
print(f'Gate 2 - Model size: {size:.0f}MB', '✅ <500MB' if size < 500 else '❌ too large')
print(f'         Bonus gate: {"✅ <250MB" if size < 250 else "❌ >250MB"}')

# Gate 3: Latency
from inference import run as _run
prompts = ['weather in Tokyo in C', '100 USD to EUR', 'convert 5 km to miles'] * 5
times = []
for p in prompts:
    t = time.time(); _run(p, []); times.append((time.time()-t)*1000)
mean_ms = sum(times)/len(times)
print(f'Gate 3 - Latency: mean={mean_ms:.1f}ms', '✅' if mean_ms < 200 else '❌ too slow')

# Gate 4: Adapter loads
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', torch_dtype=torch.float16, device_map='cpu')
PeftModel.from_pretrained(base, './artifacts/lora_adapter')
print('Gate 4 - Adapter loads: ✅')
del base